In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

# Project root
PROJECT_ROOT = Path.cwd().parent

# Data directories
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Create processed directory if it doesn't exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DIR)
print("Processed data:", PROCESSED_DIR)

Project root: E:\Thanuj_V\Projects\marketing_campaign_optimizer
Raw data: E:\Thanuj_V\Projects\marketing_campaign_optimizer\data\raw
Processed data: E:\Thanuj_V\Projects\marketing_campaign_optimizer\data\processed


In [2]:
# File paths
marketing_path = RAW_DIR / "marketing_AB.csv"
cookie_cats_path = RAW_DIR / "cookie_cats.csv"

# Load raw datasets
marketing_df = pd.read_csv(marketing_path)
cookie_cats_df = pd.read_csv(cookie_cats_path)

# Display basic information
print("Marketing A/B dataset:")
print(f"  Rows: {marketing_df.shape[0]:,}")
print(f"  Columns: {marketing_df.shape[1]}")

print("\nCookie Cats dataset:")
print(f"  Rows: {cookie_cats_df.shape[0]:,}")
print(f"  Columns: {cookie_cats_df.shape[1]}")

Marketing A/B dataset:
  Rows: 588,101
  Columns: 7

Cookie Cats dataset:
  Rows: 90,189
  Columns: 5


In [3]:
# Expected columns based on the source datasets

expected_marketing_columns = {
    "Unnamed: 0",
    "user id",
    "test group",
    "converted",
    "total ads",
    "most ads day",
    "most ads hour"
}

expected_cookie_cats_columns = {
    "userid",
    "version",
    "sum_gamerounds",
    "retention_1",
    "retention_7"
}

# Compare actual vs expected columns
marketing_columns = set(marketing_df.columns)
cookie_cats_columns = set(cookie_cats_df.columns)

print("=== MARKETING A/B SCHEMA VALIDATION ===")

missing_marketing = expected_marketing_columns - marketing_columns
unexpected_marketing = marketing_columns - expected_marketing_columns

print("Missing columns:", missing_marketing)
print("Unexpected columns:", unexpected_marketing)

print("\n=== COOKIE CATS SCHEMA VALIDATION ===")

missing_cookie = expected_cookie_cats_columns - cookie_cats_columns
unexpected_cookie = cookie_cats_columns - expected_cookie_cats_columns

print("Missing columns:", missing_cookie)
print("Unexpected columns:", unexpected_cookie)

# Fail fast if the schema is not what we expect
assert not missing_marketing, f"Marketing A/B missing columns: {missing_marketing}"
assert not missing_cookie, f"Cookie Cats missing columns: {missing_cookie}"

print("\nSchema validation PASSED.")

=== MARKETING A/B SCHEMA VALIDATION ===
Missing columns: set()
Unexpected columns: set()

=== COOKIE CATS SCHEMA VALIDATION ===
Missing columns: set()
Unexpected columns: set()

Schema validation PASSED.


In [4]:
# ============================================================
# VALUE AND DATA-TYPE VALIDATION
# ============================================================

print("=== MARKETING A/B VALIDATION ===")

# 1. Check unique users
assert marketing_df["user id"].is_unique, \
    "Marketing A/B contains duplicate user IDs."

print("✓ User IDs are unique")

# 2. Check experiment groups
expected_marketing_groups = {"ad", "psa"}
actual_marketing_groups = set(marketing_df["test group"].unique())

assert actual_marketing_groups == expected_marketing_groups, \
    f"Unexpected marketing groups: {actual_marketing_groups}"

print(f"✓ Experiment groups: {sorted(actual_marketing_groups)}")

# 3. Check conversion values
expected_conversion_values = {True, False}
actual_conversion_values = set(marketing_df["converted"].unique())

assert actual_conversion_values.issubset(expected_conversion_values), \
    f"Unexpected conversion values: {actual_conversion_values}"

print("✓ Conversion variable is binary")

# 4. Check total ads
assert (marketing_df["total ads"] >= 0).all(), \
    "Marketing dataset contains negative ad counts."

print("✓ Total ads contains no negative values")

# 5. Check advertising hour
assert marketing_df["most ads hour"].between(0, 23).all(), \
    "Marketing dataset contains invalid hours."

print("✓ Advertising hours are between 0 and 23")


print("\n=== COOKIE CATS VALIDATION ===")

# 6. Check unique users
assert cookie_cats_df["userid"].is_unique, \
    "Cookie Cats contains duplicate user IDs."

print("✓ User IDs are unique")

# 7. Check experiment versions
expected_cookie_versions = {"gate_30", "gate_40"}
actual_cookie_versions = set(cookie_cats_df["version"].unique())

assert actual_cookie_versions == expected_cookie_versions, \
    f"Unexpected Cookie Cats versions: {actual_cookie_versions}"

print(f"✓ Experiment versions: {sorted(actual_cookie_versions)}")

# 8. Check game rounds
assert (cookie_cats_df["sum_gamerounds"] >= 0).all(), \
    "Cookie Cats contains negative game rounds."

print("✓ Game rounds contains no negative values")

# 9. Check retention variables are binary
assert set(cookie_cats_df["retention_1"].unique()).issubset({True, False}), \
    "Unexpected values in retention_1."

assert set(cookie_cats_df["retention_7"].unique()).issubset({True, False}), \
    "Unexpected values in retention_7."

print("✓ Retention variables are binary")

print("\nAll value validations PASSED.")

=== MARKETING A/B VALIDATION ===
✓ User IDs are unique
✓ Experiment groups: ['ad', 'psa']
✓ Conversion variable is binary
✓ Total ads contains no negative values
✓ Advertising hours are between 0 and 23

=== COOKIE CATS VALIDATION ===
✓ User IDs are unique
✓ Experiment versions: ['gate_30', 'gate_40']
✓ Game rounds contains no negative values
✓ Retention variables are binary

All value validations PASSED.


In [5]:
# ============================================================
# CREATE CLEAN PROCESSED DATASETS
# ============================================================

# Make copies so the raw data remains untouched
marketing_clean = marketing_df.copy()
cookie_cats_clean = cookie_cats_df.copy()

# ------------------------------------------------------------
# MARKETING A/B CLEANING
# ------------------------------------------------------------

# Remove the technical CSV index column.
# It contains only a sequential row number and no analytical value.
marketing_clean = marketing_clean.drop(columns=["Unnamed: 0"])

# Standardize column names for easier SQL/Python usage
marketing_clean = marketing_clean.rename(columns={
    "user id": "user_id",
    "test group": "test_group",
    "total ads": "total_ads",
    "most ads day": "most_ads_day",
    "most ads hour": "most_ads_hour"
})

# ------------------------------------------------------------
# COOKIE CATS CLEANING
# ------------------------------------------------------------

# Cookie Cats column names are already suitable,
# so no structural changes are required.
cookie_cats_clean = cookie_cats_clean.copy()

# ------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------

print("Marketing A/B processed shape:", marketing_clean.shape)
print("Cookie Cats processed shape:", cookie_cats_clean.shape)

print("\nMarketing A/B columns:")
print(marketing_clean.columns.tolist())

print("\nCookie Cats columns:")
print(cookie_cats_clean.columns.tolist())

Marketing A/B processed shape: (588101, 6)
Cookie Cats processed shape: (90189, 5)

Marketing A/B columns:
['user_id', 'test_group', 'converted', 'total_ads', 'most_ads_day', 'most_ads_hour']

Cookie Cats columns:
['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7']


In [6]:
# ============================================================
# SAVE PROCESSED DATASETS
# ============================================================

marketing_processed_path = PROCESSED_DIR / "marketing_ab_processed.csv"
cookie_cats_processed_path = PROCESSED_DIR / "cookie_cats_processed.csv"

# Save cleaned datasets
marketing_clean.to_csv(marketing_processed_path, index=False)
cookie_cats_clean.to_csv(cookie_cats_processed_path, index=False)

print("Processed datasets saved successfully:")
print("Marketing A/B:", marketing_processed_path)
print("Cookie Cats:", cookie_cats_processed_path)

Processed datasets saved successfully:
Marketing A/B: E:\Thanuj_V\Projects\marketing_campaign_optimizer\data\processed\marketing_ab_processed.csv
Cookie Cats: E:\Thanuj_V\Projects\marketing_campaign_optimizer\data\processed\cookie_cats_processed.csv


In [7]:
# ============================================================
# VERIFY PROCESSED FILES
# ============================================================

# Load the saved processed files again
marketing_check = pd.read_csv(marketing_processed_path)
cookie_cats_check = pd.read_csv(cookie_cats_processed_path)

# Check shapes
print("=== SHAPE CHECK ===")

print("Marketing A/B:")
print("  Original:", marketing_df.shape)
print("  Processed:", marketing_check.shape)

print("\nCookie Cats:")
print("  Original:", cookie_cats_df.shape)
print("  Processed:", cookie_cats_check.shape)


# ------------------------------------------------------------
# INTEGRITY CHECKS
# ------------------------------------------------------------

# Marketing: only the technical index column should have been removed
assert marketing_check.shape[0] == marketing_df.shape[0], \
    "Marketing row count changed."

assert marketing_check.shape[1] == marketing_df.shape[1] - 1, \
    "Marketing column count is unexpected."

# Cookie Cats: no rows or columns should have changed
assert cookie_cats_check.shape == cookie_cats_df.shape, \
    "Cookie Cats shape changed."

# Check that user IDs are still unique
assert marketing_check["user_id"].is_unique, \
    "Marketing user IDs are no longer unique."

assert cookie_cats_check["userid"].is_unique, \
    "Cookie Cats user IDs are no longer unique."


print("\nAll processed-file integrity checks PASSED.")

=== SHAPE CHECK ===
Marketing A/B:
  Original: (588101, 7)
  Processed: (588101, 6)

Cookie Cats:
  Original: (90189, 5)
  Processed: (90189, 5)

All processed-file integrity checks PASSED.


In [8]:
# ============================================================
# MISSING VALUE CHECK
# ============================================================

print("=== MARKETING A/B MISSING VALUES ===")
print(marketing_check.isnull().sum())

print("\n=== COOKIE CATS MISSING VALUES ===")
print(cookie_cats_check.isnull().sum())


# Fail if any missing values exist
assert marketing_check.isnull().sum().sum() == 0, \
    "Marketing A/B contains missing values."

assert cookie_cats_check.isnull().sum().sum() == 0, \
    "Cookie Cats contains missing values."


print("\nNo missing values found. Validation PASSED.")

=== MARKETING A/B MISSING VALUES ===
user_id          0
test_group       0
converted        0
total_ads        0
most_ads_day     0
most_ads_hour    0
dtype: int64

=== COOKIE CATS MISSING VALUES ===
userid            0
version           0
sum_gamerounds    0
retention_1       0
retention_7       0
dtype: int64

No missing values found. Validation PASSED.


In [9]:
# ============================================================
# DUPLICATE ROW CHECK
# ============================================================

print("=== DUPLICATE ROW CHECK ===")

marketing_duplicates = marketing_check.duplicated().sum()
cookie_cats_duplicates = cookie_cats_check.duplicated().sum()

print(f"Marketing A/B duplicate rows: {marketing_duplicates:,}")
print(f"Cookie Cats duplicate rows: {cookie_cats_duplicates:,}")


# Fail if duplicate rows exist
assert marketing_duplicates == 0, \
    "Marketing A/B contains duplicate rows."

assert cookie_cats_duplicates == 0, \
    "Cookie Cats contains duplicate rows."


print("\nNo duplicate rows found. Validation PASSED.")

=== DUPLICATE ROW CHECK ===
Marketing A/B duplicate rows: 0
Cookie Cats duplicate rows: 0

No duplicate rows found. Validation PASSED.


In [10]:
# ============================================================
# FINAL DATA SANITY CHECK
# ============================================================

print("=== MARKETING A/B SAMPLE ===")
display(marketing_check.head())

print("\n=== MARKETING A/B DATA TYPES ===")
print(marketing_check.dtypes)

print("\n=== COOKIE CATS SAMPLE ===")
display(cookie_cats_check.head())

print("\n=== COOKIE CATS DATA TYPES ===")
print(cookie_cats_check.dtypes)

print("\n=== MARKETING A/B SUMMARY ===")
display(marketing_check.describe(include="all"))

print("\n=== COOKIE CATS SUMMARY ===")
display(cookie_cats_check.describe(include="all"))

=== MARKETING A/B SAMPLE ===


,user_id,test_group,converted,total_ads,most_ads_day,most_ads_hour
0,1069124,ad,False,130,Monday,20
1,1119715,ad,False,93,Tuesday,22
2,1144181,ad,False,21,Tuesday,18
3,1435133,ad,False,355,Tuesday,10
4,1015700,ad,False,276,Friday,14



=== MARKETING A/B DATA TYPES ===
user_id          int64
test_group         str
converted         bool
total_ads        int64
most_ads_day       str
most_ads_hour    int64
dtype: object

=== COOKIE CATS SAMPLE ===


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True



=== COOKIE CATS DATA TYPES ===
userid            int64
version             str
sum_gamerounds    int64
retention_1        bool
retention_7        bool
dtype: object

=== MARKETING A/B SUMMARY ===


,user_id,test_group,converted,total_ads,most_ads_day,most_ads_hour
count,5.881010e+05,588101,588101,588101.000000,588101,588101.000000
unique,NaN,2,2,NaN,7,NaN
top,NaN,ad,False,NaN,Friday,NaN
freq,NaN,564577,573258,NaN,92608,NaN
mean,1.310692e+06,NaN,NaN,24.820876,NaN,14.469061
std,2.022260e+05,NaN,NaN,43.715181,NaN,4.834634
min,9.000000e+05,NaN,NaN,1.000000,NaN,0.000000
25%,1.143190e+06,NaN,NaN,4.000000,NaN,11.000000
50%,1.313725e+06,NaN,NaN,13.000000,NaN,14.000000
75%,1.484088e+06,NaN,NaN,27.000000,NaN,18.000000



=== COOKIE CATS SUMMARY ===


,userid,version,sum_gamerounds,retention_1,retention_7
count,9.018900e+04,90189,90189.000000,90189,90189
unique,NaN,2,NaN,2,2
top,NaN,gate_40,NaN,False,False
freq,NaN,45489,NaN,50036,73408
mean,4.998412e+06,NaN,51.872457,NaN,NaN
std,2.883286e+06,NaN,195.050858,NaN,NaN
min,1.160000e+02,NaN,0.000000,NaN,NaN
25%,2.512230e+06,NaN,5.000000,NaN,NaN
50%,4.995815e+06,NaN,16.000000,NaN,NaN
75%,7.496452e+06,NaN,51.000000,NaN,NaN


In [11]:
# ============================================================
# DATA INGESTION COMPLETE
# ============================================================

print("=" * 60)
print("DATA INGESTION AND VALIDATION COMPLETE")
print("=" * 60)

print("\nMarketing A/B Testing")
print(f"Rows: {marketing_check.shape[0]:,}")
print(f"Columns: {marketing_check.shape[1]}")
print(f"Groups: {sorted(marketing_check['test_group'].unique())}")

print("\nCookie Cats")
print(f"Rows: {cookie_cats_check.shape[0]:,}")
print(f"Columns: {cookie_cats_check.shape[1]}")
print(f"Versions: {sorted(cookie_cats_check['version'].unique())}")

print("\nQuality checks:")
print("✓ Schema validation")
print("✓ Value validation")
print("✓ Row-count integrity")
print("✓ User ID uniqueness")
print("✓ Missing-value check")
print("✓ Duplicate-row check")
print("✓ Processed files verified")

print("\nSTATUS: READY FOR EDA")

DATA INGESTION AND VALIDATION COMPLETE

Marketing A/B Testing
Rows: 588,101
Columns: 6
Groups: ['ad', 'psa']

Cookie Cats
Rows: 90,189
Columns: 5
Versions: ['gate_30', 'gate_40']

Quality checks:
✓ Schema validation
✓ Value validation
✓ Row-count integrity
✓ User ID uniqueness
✓ Missing-value check
✓ Duplicate-row check
✓ Processed files verified

STATUS: READY FOR EDA
